# Personal Protective Equipment (PPE) Detection System
## End-to-End YOLO26n Training, Checkpoint Resumption & Authenticity Enforcement Pipeline
**Approved Target Classes (6)**: `0: person`, `1: helmet`, `2: safety_vest`, `3: gloves`, `4: safety_shoes`, `5: goggles`  
**Dataset**: 3,252 Annotated Industrial PPE Images (70% Train / 20% Val / 10% Held-Out Test)  
**Model Architecture**: YOLO26n (2.5M Parameters, strictly initialized from scratch)

### 1. Project Setup & Dynamic Path Configuration

In [ ]:
import os
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "Notebooks" else CURRENT_DIR

DATASETS_DIR = PROJECT_ROOT / "Datasets"
PROCESSED_DIR = DATASETS_DIR / "Processed" / "PPE_Final"
DATA_YAML = PROCESSED_DIR / "data.yaml"
OUTPUT_DIR = PROJECT_ROOT / "Output"
MODEL_DIR = PROJECT_ROOT / "Model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["person", "helmet", "safety_vest", "gloves", "safety_shoes", "goggles"]

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset YAML: {DATA_YAML}")
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

### 2. Hardware Diagnostics (CPU & CUDA GPU Detection)

In [ ]:
import torch
from ultralytics import YOLO

if torch.cuda.is_available():
    DEVICE = 0
    BATCH_SIZE = 16
    WORKERS = 4
    print(f"CUDA GPU Active: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    BATCH_SIZE = 8
    WORKERS = 0
    print("Running on CPU (12th Gen Intel Core i5-1235U, 12 threads)")

print(f"PyTorch Version: {torch.__version__}")
print(f"Selected Device: {DEVICE} | Batch Size: {BATCH_SIZE}")

### 3. Verify Dataset Configuration & Class Splits

In [ ]:
import yaml

with open(DATA_YAML, "r") as f:
    yaml_config = yaml.safe_load(f)

print("=" * 60)
print("DATASET YAML CONFIGURATION:")
print(f"Train Path: {yaml_config.get('train')}")
print(f"Val Path:   {yaml_config.get('val')}")
print(f"Test Path:  {yaml_config.get('test')}")
print(f"Class Map:  {yaml_config.get('names')}")
print("=" * 60)

### 4. Baseline Training Execution (Epoch 1 From Scratch)

In [ ]:
# Initialize YOLO26n with strictly random weights (pretrained=False)
model_scratch = YOLO("yolo26n.yaml")

print("Launching Baseline Epoch 1 Training from Scratch...")
baseline_results = model_scratch.train(
    data=str(DATA_YAML),
    epochs=1,
    imgsz=640,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="PPE_YOLO26n_From_Scratch",
    patience=30,
    plots=True,
    save=True,
    workers=WORKERS,
    seed=42,
    exist_ok=True
)
print("Baseline Epoch 1 Completed!")

### 5. Checkpoint Resumption (Resuming from `last.pt` for Epoch 2+)
Resumes training using Ultralytics native checkpoint restoration, restoring neural weights, optimizer momentum buffers, and learning rate trajectories.

In [ ]:
# Resolve checkpoint path
resume_ckpt = OUTPUT_DIR / "PPE_Resumed_Training" / "weights" / "last.pt"
if not resume_ckpt.exists():
    resume_ckpt = OUTPUT_DIR / "PPE_YOLO26n_From_Scratch" / "weights" / "last.pt"

print(f"Loading checkpoint for resumption: {resume_ckpt}")
model_resume = YOLO(str(resume_ckpt))

# Resume training for target epochs (e.g. 2 for 2nd epoch, or 10-50 for extended learning)
RESUME_EPOCHS = 2

resume_results = model_resume.train(
    data=str(DATA_YAML),
    epochs=RESUME_EPOCHS,
    resume=True,
    imgsz=640,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="PPE_Resumed_Training",
    patience=30,
    workers=WORKERS,
    save=True,
    plots=True,
    exist_ok=True
)
print("Resumed Training Epoch Completed!")

### 6. Production Multi-Epoch Extended Training (50–200 Epochs)
For full production-grade convergence across all 6 classes, use the extended training routine below with AdamW optimizer, cosine annealing, and early stopping.

In [ ]:
# Production Training Configuration
EXTENDED_EPOCHS = 50  # Set to 50, 100, or 200 epochs

prod_ckpt = OUTPUT_DIR / "PPE_Resumed_Training" / "weights" / "best.pt"
if not prod_ckpt.exists():
    prod_ckpt = "yolo26n.yaml"

print(f"Starting weights: {prod_ckpt}")
prod_model = YOLO(str(prod_ckpt))

prod_results = prod_model.train(
    data=str(DATA_YAML),
    epochs=EXTENDED_EPOCHS,
    imgsz=640,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="PPE_Production_Training",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=25,
    save=True,
    plots=True,
    seed=42,
    exist_ok=True,
    workers=WORKERS
)

print(f"Production training complete! Best model: {OUTPUT_DIR / 'PPE_Production_Training' / 'weights' / 'best.pt'}")

### 7. Per-Class Performance Breakdown & Metric Visualizations
Compare class metrics and loss curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

resumed_results = OUTPUT_DIR / "PPE_Resumed_Training" / "results.csv"
if resumed_results.exists():
    df = pd.read_csv(resumed_results)
    df.columns = [c.strip() for c in df.columns]
    print("=" * 70)
    print("TRAINING METRIC TRAJECTORY:")
    print("=" * 70)
    cols = ['epoch', 'train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss', 'metrics/mAP50(B)']
    avail = [c for c in cols if c in df.columns]
    print(df[avail].to_string(index=False))

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
res_plot = OUTPUT_DIR / "PPE_Resumed_Training" / "results.png"
if res_plot.exists():
    axes[0].imshow(mpimg.imread(str(res_plot)))
    axes[0].set_title("Training Loss & Metric Curves", fontweight="bold")
    axes[0].axis("off")

cm_plot = OUTPUT_DIR / "PPE_Resumed_Training" / "confusion_matrix_normalized.png"
if cm_plot.exists():
    axes[1].imshow(mpimg.imread(str(cm_plot)))
    axes[1].set_title("Normalized Confusion Matrix", fontweight="bold")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

### 8. Multi-Class PPE Authenticity Enforcement
Eliminates false positives:
- **`helmet`**: Enforces 3D dome aspect ratio ($0.45 \le h/w \le 1.85$). Filters flat tissue paper. Requires conf $\ge 0.18$ for white objects.
- **`safety_vest`**: Enforces saturation $S \ge 30$ and fluorescent yellow/lime or neon orange hues. Discards plain black, grey, and cotton t-shirts.
- **`gloves`**: Excludes bare human skin using YCrCb chrominance segmentation.
- **`safety_shoes`**: Filters flat slippers ($h/w < 0.35$) and bare feet; accepts heavy work boots.
- **`goggles`**: Validates eye-line positioning ($10\% - 60\%$ down face) and face span.

In [ ]:
import cv2
import numpy as np

def validate_ppe_candidate(
    cname: str,
    frame: np.ndarray,
    box: list,
    conf: float,
    face_box: list = None
) -> tuple[bool, str]:
    x1, y1, x2, y2 = [int(v) for v in box]
    frame_h, frame_w = frame.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame_w, x2), min(frame_h, y2)

    bw = x2 - x1
    bh = y2 - y1
    if bw < 8 or bh < 8:
        return False, "Box too small"

    aspect_ratio = bh / max(1.0, float(bw))
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return False, "Empty crop"

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    mean_h = float(np.mean(hsv[:, :, 0]))
    mean_s = float(np.mean(hsv[:, :, 1]))
    mean_v = float(np.mean(hsv[:, :, 2]))

    # 1. HELMET
    if cname == "helmet":
        if aspect_ratio < 0.45:
            return False, f"Aspect ratio too flat ({aspect_ratio:.2f} < 0.45) - likely paper/tissue"
        if aspect_ratio > 1.85:
            return False, f"Aspect ratio too tall ({aspect_ratio:.2f} > 1.85)"

        is_colored = (mean_s >= 25.0)
        if not is_colored:
            if conf < 0.18:
                return False, f"White candidate confidence ({conf:.2f} < 0.18) too low - rejected tissue/paper"
        else:
            if conf < 0.08:
                return False, f"Helmet confidence too low ({conf:.2f} < 0.08)"

        if face_box is not None:
            fx1, fy1, fx2, fy2 = face_box
            fw = max(1.0, float(fx2 - fx1))
            fh = max(1.0, float(fy2 - fy1))
            if bw / fw < 0.65 or bw / fw > 2.50:
                return False, f"Helmet width ratio ({bw/fw:.2f}) invalid for head"
            h_cx = (x1 + x2) / 2.0
            f_cx = (fx1 + fx2) / 2.0
            if abs(h_cx - f_cx) / fw > 0.55:
                return False, "Helmet not centered on head"
            if (fy1 - y1) / fh < 0.12:
                return False, "Insufficient dome height above forehead"
            if (y2 - fy1) / fh > 0.50:
                return False, "Helmet extends too far down face"

        return True, "Valid industrial helmet"

    # 2. SAFETY VEST
    elif cname == "safety_vest":
        if mean_s < 30.0 and mean_v < 180.0:
            return False, f"Lacks high-visibility safety coloration (saturation {mean_s:.1f} < 30.0) - ordinary clothing"

        is_fluorescent_hue = (20 <= mean_h <= 75) or (mean_h <= 24) or (mean_h >= 170)
        if not is_fluorescent_hue:
            if conf < 0.18:
                return False, f"Non-fluorescent clothing confidence ({conf:.2f} < 0.18) too low for safety vest"

        if aspect_ratio < 0.55 or aspect_ratio > 2.60:
            return False, f"Vest aspect ratio ({aspect_ratio:.2f}) invalid"

        if conf < 0.08:
            return False, f"Safety vest confidence too low ({conf:.2f} < 0.08)"

        return True, "Valid high-vis safety vest"

    # 3. GLOVES
    elif cname == "gloves":
        if conf < 0.12:
            return False, f"Gloves confidence too low ({conf:.2f} < 0.12)"

        ycrcb = cv2.cvtColor(crop, cv2.COLOR_BGR2YCrCb)
        cr = ycrcb[:, :, 1]
        cb = ycrcb[:, :, 2]
        skin_mask = (cr >= 133) & (cr <= 173) & (cb >= 77) & (cb <= 127)
        skin_ratio = float(np.mean(skin_mask))

        if skin_ratio > 0.65:
            return False, f"Detected bare human skin ({skin_ratio:.1%}) - not a safety glove"

        return True, "Valid industrial safety glove"

    # 4. SAFETY SHOES
    elif cname == "safety_shoes":
        if conf < 0.12:
            return False, f"Safety shoes confidence too low ({conf:.2f} < 0.12)"

        if aspect_ratio < 0.35:
            return False, f"Shoe aspect ratio too flat ({aspect_ratio:.2f} < 0.35) - likely slippers/bare foot"

        ycrcb = cv2.cvtColor(crop, cv2.COLOR_BGR2YCrCb)
        cr = ycrcb[:, :, 1]
        cb = ycrcb[:, :, 2]
        skin_mask = (cr >= 133) & (cr <= 173) & (cb >= 77) & (cb <= 127)
        skin_ratio = float(np.mean(skin_mask))
        if skin_ratio > 0.65:
            return False, f"Detected bare human foot skin ({skin_ratio:.1%}) - not a safety shoe"

        return True, "Valid industrial safety shoe"

    # 5. GOGGLES
    elif cname == "goggles":
        if conf < 0.12:
            return False, f"Goggles confidence too low ({conf:.2f} < 0.12)"

        if face_box is not None:
            fx1, fy1, fx2, fy2 = face_box
            fw = max(1.0, float(fx2 - fx1))
            fh = max(1.0, float(fy2 - fy1))
            g_cx = (x1 + x2) / 2.0
            g_cy = (y1 + y2) / 2.0
            f_cx = (fx1 + fx2) / 2.0

            y_rel = (g_cy - fy1) / fh
            if y_rel < 0.10 or y_rel > 0.60:
                return False, f"Goggles not at eye line (relative Y {y_rel:.2f} outside 0.10-0.60)"

            if abs(g_cx - f_cx) / fw > 0.40:
                return False, "Goggles not centered across eyes"

            if bw / fw < 0.40 or bw / fw > 1.25:
                return False, f"Goggles width ratio ({bw/fw:.2f}) invalid for eyes"

        return True, "Valid protective safety goggles"

    return True, "Valid detection"

print("Multi-Class Authenticity Verification Engine Ready!")

### 9. Test Split Inference & Visual Side-by-Side Comparison
Visualizes raw YOLO predictions vs. authenticity-filtered predictions on held-out test split images.

In [ ]:
import matplotlib.patches as patches

best_model_path = MODEL_DIR / "PPE_YOLO26n_Resumed_Best.pt"
test_model = YOLO(str(best_model_path))

test_img_dir = PROCESSED_DIR / "images" / "test"
test_imgs = sorted(list(test_img_dir.glob("*.jpg")))

if test_imgs:
    sample_p = test_imgs[0]
    img_bgr = cv2.imread(str(sample_p))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Run YOLO detection
    results = test_model.predict(img_bgr, conf=0.08, verbose=False)[0]
    
    raw_list = []
    filtered_list = []
    
    for b in results.boxes:
        cid = int(b.cls[0].item())
        cname = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f"cls_{cid}"
        conf = float(b.conf[0].item())
        box = [float(v) for v in b.xyxy[0].tolist()]
        
        raw_list.append((cname, conf, box))
        
        if cname == "person":
            filtered_list.append((cname, conf, box))
        else:
            is_valid, _ = validate_ppe_candidate(cname, img_bgr, box, conf)
            if is_valid:
                filtered_list.append((cname, conf, box))
                
    # Side-by-side visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Raw YOLO Detections (Count: {len(raw_list)})", fontsize=12, fontweight="bold")
    axes[0].axis("off")
    for cname, conf, box in raw_list:
        bx1, by1, bx2, by2 = box
        rect = patches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2, edgecolor="red", facecolor="none")
        axes[0].add_patch(rect)
        axes[0].text(bx1, by1 - 4, f"{cname} {conf:.2f}", color="white", fontsize=9, bbox=dict(facecolor="red", alpha=0.7, pad=1))
        
    axes[1].imshow(img_rgb)
    axes[1].set_title(f"Authenticity Verified Detections (Count: {len(filtered_list)})", fontsize=12, fontweight="bold")
    axes[1].axis("off")
    for cname, conf, box in filtered_list:
        bx1, by1, bx2, by2 = box
        rect = patches.Rectangle((bx1, by1), bx2 - bx1, by2 - by1, linewidth=2, edgecolor="green", facecolor="none")
        axes[1].add_patch(rect)
        axes[1].text(bx1, by1 - 4, f"{cname} {conf:.2f}", color="white", fontsize=9, bbox=dict(facecolor="green", alpha=0.7, pad=1))
        
    plt.tight_layout()
    plt.show()

### 10. Model Export & Live Production Deployment
Verify that the model checkpoint is exported to `Model/PPE_YOLO26n_Resumed_Best.pt` for live webcam detection.

In [ ]:
import shutil

source_pt = OUTPUT_DIR / "PPE_Resumed_Training" / "weights" / "best.pt"
target_pt = MODEL_DIR / "PPE_YOLO26n_Resumed_Best.pt"

if source_pt.exists():
    shutil.copy2(source_pt, target_pt)
    print("=" * 65)
    print("DEPLOYMENT MODEL VERIFICATION:")
    print("=" * 65)
    print(f"Exported Path: {target_pt}")
    print(f"File Size:     {target_pt.stat().st_size / 1e6:.2f} MB")
    print(f"Classes:       {CLASS_NAMES}")
    print("STATUS: Verified and ready for live webcam detection!")
    print("=" * 65)
else:
    print(f"Source weights not found at: {source_pt}")